# OpenEnv SME Negotiator - GRPO Training on Colab

This notebook is a thin Colab wrapper over the canonical environment-native trainer in `rl/train_grpo_trl.py`.

What it does:
1. Installs the repo and RL dependencies.
2. Smoke-tests the environment and records a heuristic baseline.
3. Builds the canonical training session with shared prompt, tokenizer, environment, reward, and callback wiring.
4. Runs GRPO with model-only checkpoint saving.
5. Plots training diagnostics and compares a trained policy transcript against the heuristic baseline.

Design note: the environment step loop lives in repo code, not in notebook cells. The notebook is for orchestration, visibility, and quick Colab iteration.

## 1. Install the repo and RL dependencies

The Colab path should use the same importable modules as local development so the notebook does not drift away from the canonical trainer.

In [ ]:
%pip install -q -U pip

from pathlib import Path

REPO_URL = "https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git"
REPO_DIR = Path("OpenEnv_SME_Negotiator-2.o")

if not REPO_DIR.exists():
    !git clone -q $REPO_URL

%cd $REPO_DIR
%pip install -q "trl==0.29.0" "transformers==5.5.0" "peft>=0.19.1" "datasets" "accelerate" "matplotlib" "huggingface_hub>=0.20" "openenv-core>=0.2.3"
%pip install -q -e .[rl]

## 2. Colab configuration

These defaults are intentionally T4-safe and match the canonical environment-native trainer instead of re-implementing training logic in the notebook.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["OPENAI_API_KEY"] = hf_token
        os.environ["OPENAI_BASE_URL"] = "https://router.huggingface.co/v1"
        os.environ["API_BASE_URL"] = "https://router.huggingface.co/v1"
except Exception:
    pass

RUN_PROFILE = "submission"
MODEL_NAME = "Qwen/Qwen3-0.6B"
TASK_NAME = "liquidity-correlation-hard"
DIFFICULTY = "hard"
TOTAL_PERIODS = 2
SEED_BASE = 1000
MAX_EPISODE_STEPS = 12
NUM_GENERATIONS = 4
MAX_COMPLETION_LENGTH = 128
PROFILE_DEFAULTS = {
    "submission": {"num_samples": 64, "train_steps": 100},
    "smoke": {"num_samples": 24, "train_steps": 12},
}
if RUN_PROFILE not in PROFILE_DEFAULTS:
    raise ValueError(f"Unsupported RUN_PROFILE: {RUN_PROFILE!r}")
NUM_SAMPLES = PROFILE_DEFAULTS[RUN_PROFILE]["num_samples"]
TRAIN_STEPS = PROFILE_DEFAULTS[RUN_PROFILE]["train_steps"]
EVAL_SEEDS = [1000, 1001, 1002, 1003]
USE_VLLM = False
OUTPUT_DIR = Path("outputs/grpo_sme_liquidity_colab")
REWARD_CURVE_PATH = OUTPUT_DIR / "reward_curve.png"
TRAINING_DASHBOARD_PATH = OUTPUT_DIR / "training_dashboard.png"
EVAL_SUMMARY_PATH = OUTPUT_DIR / "eval_summary.json"
POLICY_COMPARISON_PATH = OUTPUT_DIR / "policy_comparison.png"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

{
    "RUN_PROFILE": RUN_PROFILE,
    "MODEL_NAME": MODEL_NAME,
    "TASK_NAME": TASK_NAME,
    "DIFFICULTY": DIFFICULTY,
    "TOTAL_PERIODS": TOTAL_PERIODS,
    "NUM_SAMPLES": NUM_SAMPLES,
    "SEED_BASE": SEED_BASE,
    "MAX_EPISODE_STEPS": MAX_EPISODE_STEPS,
    "NUM_GENERATIONS": NUM_GENERATIONS,
    "MAX_COMPLETION_LENGTH": MAX_COMPLETION_LENGTH,
    "TRAIN_STEPS": TRAIN_STEPS,
    "EVAL_SEEDS": EVAL_SEEDS,
    "USE_VLLM": USE_VLLM,
    "OUTPUT_DIR": str(OUTPUT_DIR),
}

## 3. Environment smoke test

OpenEnv training still starts with a normal environment loop. This quick check confirms `reset()` and `step()` behave before we spend GPU time on GRPO.

In [ ]:
from server.environment import SMELiquidityEnvironment
from sme_negotiator_env.models import NegotiationAction

env_smoke = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
obs = env_smoke.reset(seed=42, difficulty=DIFFICULTY, task_name=TASK_NAME)
print({
    "task_name": obs.task_name,
    "buyer_price": obs.buyer_price,
    "buyer_days": obs.buyer_days,
    "liquidity_threshold": obs.liquidity_threshold,
    "open_deals": obs.open_deal_ids,
})

next_obs = env_smoke.step(NegotiationAction(action_type="advance_period"))
print({
    "current_period": next_obs.current_period,
    "done": next_obs.done,
    "reward": next_obs.reward,
})

## 4. Heuristic baseline

We keep a deterministic baseline run so the notebook always shows a before/after reference point alongside training diagnostics.

In [ ]:
from statistics import mean

from rl.demo import run_heuristic_episode

BASELINE_SEEDS = list(EVAL_SEEDS)
baseline_runs = [
    run_heuristic_episode(
        seed=seed,
        total_periods=TOTAL_PERIODS,
        task_name=TASK_NAME,
        difficulty=DIFFICULTY,
        max_steps=MAX_EPISODE_STEPS,
    )
    for seed in BASELINE_SEEDS
]

def _baseline_success(run):
    summary = run.get("summary", {})
    if isinstance(summary, dict):
        return bool(summary.get("success_no_default_positive_npv", False))
    return bool(getattr(summary, "success_no_default_positive_npv", False))

baseline_summary = {
    "mean_total_reward": mean(run["total_reward"] for run in baseline_runs),
    "mean_steps": mean(run["steps"] for run in baseline_runs),
    "success_rate": mean(1.0 if _baseline_success(run) else 0.0 for run in baseline_runs),
}
baseline_summary

## 5. Shared action contract preview

The notebook previews the exact prompt contract used by training, but the environment/tool interaction itself still comes from the shared trainer code.

In [ ]:
from rl.bridge import SUPPORTED_ACTION_TYPES, SUPPORTED_TOOL_NAMES, build_action_contract_text
from rl.train_grpo_trl import DEFAULT_PROMPT

print("Default prompt:\n")
print(DEFAULT_PROMPT)
print("\nSupported action types:", SUPPORTED_ACTION_TYPES)
print("Supported tool names:", SUPPORTED_TOOL_NAMES)
print("\nStrict action contract:\n")
print(build_action_contract_text())

## 6. Build the canonical training session

This cell does the important wiring once: shared dataset rows, environment-aware tokenizer setup, LoRA-only training setup, canonical reward function, environment diagnostics, and model-only checkpoint policy.

In [ ]:
from rl.train_grpo_trl import build_training_session, make_training_args

NOTEBOOK_ARGS = make_training_args(
    model_name=MODEL_NAME,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    total_periods=TOTAL_PERIODS,
    num_samples=NUM_SAMPLES,
    seed_base=SEED_BASE,
    output_dir=str(OUTPUT_DIR),
    max_episode_steps=MAX_EPISODE_STEPS,
    num_generations=NUM_GENERATIONS,
    generation_batch_size=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=TRAIN_STEPS,
    save_steps=1000,
    use_vllm=USE_VLLM,
)

session = build_training_session(NOTEBOOK_ARGS)

{
    "dataset_rows": len(session["dataset"]),
    "output_dir": NOTEBOOK_ARGS.output_dir,
    "final_checkpoint_path": str(session["final_checkpoint_path"]),
    "num_generations": session["training_args"].num_generations,
    "max_completion_length": session["training_args"].max_completion_length,
    "max_steps": getattr(session["training_args"], "max_steps", None),
    "save_only_model": getattr(session["training_args"], "save_only_model", None),
    "tokenizer_has_response_schema": getattr(session["tokenizer"], "response_schema", None) is not None,
}

## 7. Train with the shared environment-native path

The notebook builds `GRPOTrainer` from the returned session bundle and saves only model weights at the end of training.

In [ ]:
from rl.train_grpo_trl import run_training_session

training_run = run_training_session(NOTEBOOK_ARGS)
session = training_run["session"]
trainer = training_run["trainer"]
checkpoint_path = training_run["checkpoint_path"]

print(f"Final checkpoint saved to: {Path(checkpoint_path).resolve()}")
print(f"Reward curve path: {REWARD_CURVE_PATH.resolve()}")

## 8. Diagnostics and reward curves

The canonical callback already emits zero-variance warnings during training. This cell summarizes the logged environment metrics so we can verify that GRPO is seeing non-trivial reward and action variance.

In [ ]:
from rl.demo import save_training_dashboard

dashboard = save_training_dashboard(trainer, output_dir=str(OUTPUT_DIR))
dashboard

print(f"Training dashboard path: {dashboard['training_dashboard_path']}")
print(f"Reward curve path: {dashboard['reward_curve_path']}")
if dashboard["zero_variance_warning"]:
    print("[TRAINING][WARN] environment reward variance is zero for consecutive logs. The canonical trainer callback prints sampled completion details during training when this happens.")

## 9. Before/after transcript comparison

Instead of maintaining a second notebook-only inference loop, reuse the repo helpers for fixed-seed base-vs-trained evaluation plus an optional heuristic reference transcript.

In [ ]:
from rl.demo import evaluate_before_after_policies

# Shared helper runs policy="base" and policy="trained" on the same fixed seed panel.
evaluation = evaluate_before_after_policies(
    output_dir=str(OUTPUT_DIR),
    seeds=EVAL_SEEDS,
    total_periods=TOTAL_PERIODS,
    task_name=TASK_NAME,
    difficulty=DIFFICULTY,
    checkpoint_path=str(checkpoint_path),
    model_name=MODEL_NAME,
    max_steps=MAX_EPISODE_STEPS,
    include_heuristic=True,
)

print(f"Eval summary path: {evaluation['eval_summary_path']}")
print(f"Policy comparison path: {evaluation['policy_comparison_path']}")
print(f"Submission ready: {evaluation['summary']['metadata']['submission_ready']}")
evaluation['summary']

print("Base model transcript:\n")
print(evaluation['base_transcript'][:3000])
print("\n" + "=" * 80 + "\n")
print("Trained transcript:\n")
print(evaluation['trained_transcript'][:3000])
print("\n" + "=" * 80 + "\n")
print("Heuristic reference transcript:\n")
print(evaluation['heuristic_transcript'][:3000])

## 10. Optional: publish the final checkpoint

Set `HF_REPO_ID` to your own namespace before running this cell.

In [ ]:
from huggingface_hub import HfApi

HF_REPO_ID = "YOUR_USERNAME/openenv-sme-negotiator-grpo"

if not HF_REPO_ID.startswith("YOUR_USERNAME/"):
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(checkpoint_path),
    )
    print(f"Uploaded {checkpoint_path} to {HF_REPO_ID}")
else:
    print("Set HF_REPO_ID before publishing.")